# Phase 3 — Bronze Layer (Raw → Delta)
**Apex Retail Intelligence | Celebal Technologies CEI'26 Major Project**

Reads Landing Parquet, writes Delta tables in the Bronze zone with an `ingested_at` audit
column, and appends incremental batches without deduplication (Bronze is append-only by
design — dedup and business rules belong to Silver).

In [0]:
dbutils.widgets.text("pipeline_root", "/Volumes/apex_retail1/pipeline/data", "Pipeline Storage Root")
dbutils.widgets.text("bronze_catalog", "apex_retail1", "Unity Catalog Name")
dbutils.widgets.text("bronze_schema", "bronze_tables", "Bronze Schema Name")

PIPELINE_ROOT = dbutils.widgets.get("pipeline_root")
CATALOG = dbutils.widgets.get("bronze_catalog")
SCHEMA = dbutils.widgets.get("bronze_schema")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

In [0]:
from pyspark.sql import functions as F

ENTITIES = ["customer", "product", "sales"]
LOAD_TYPES = ["historical", "incremental"]

def bronze_table_name(entity):
    return f"{CATALOG}.{SCHEMA}.bronze_{entity}"

## Delta Lake Integration + Metadata Injection + Incremental Append
For each entity (customer/product/sales):
1. Read the **historical** Landing Parquet, tag `ingested_at`, and write/overwrite the Delta table
   (first run — establishes the append-only Bronze table).
2. Read the **incremental** Landing Parquet, tag `ingested_at`, and **append** to the same Delta
   table. Bronze keeps every row from every batch — no dedup, no filtering.

In [0]:
bronze_summary = []

for entity in ENTITIES:
    table = bronze_table_name(entity)
    hist_path = f"{PIPELINE_ROOT}/landing/{entity}/historical"
    inc_path = f"{PIPELINE_ROOT}/landing/{entity}/incremental"

    # --- Historical: first load establishes the Bronze table ---
    df_hist = spark.read.parquet(hist_path).withColumn("ingested_at", F.current_timestamp()) \
        .withColumn("source_batch", F.lit("historical"))

    (
        df_hist.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    hist_rows = spark.table(table).count()
    bronze_summary.append((entity, "historical", hist_rows))
    print(f"[BRONZE] {table}: historical load -> {hist_rows} rows (table created/reset)")

    # --- Incremental: append-only, no dedup at Bronze ---
    df_inc = spark.read.parquet(inc_path).withColumn("ingested_at", F.current_timestamp()) \
        .withColumn("source_batch", F.lit("incremental"))

    (
        df_inc.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table)
    )
    total_rows = spark.table(table).count()
    inc_rows = total_rows - hist_rows
    bronze_summary.append((entity, "incremental_appended", inc_rows))
    print(f"[BRONZE] {table}: incremental append -> +{inc_rows} rows (table total = {total_rows})")

[BRONZE] apex_retail1.bronze_tables.bronze_customer: historical load -> 1052 rows (table created/reset)
[BRONZE] apex_retail1.bronze_tables.bronze_customer: incremental append -> +1053 rows (table total = 2105)
[BRONZE] apex_retail1.bronze_tables.bronze_product: historical load -> 1043 rows (table created/reset)
[BRONZE] apex_retail1.bronze_tables.bronze_product: incremental append -> +1041 rows (table total = 2084)
[BRONZE] apex_retail1.bronze_tables.bronze_sales: historical load -> 1002 rows (table created/reset)
[BRONZE] apex_retail1.bronze_tables.bronze_sales: incremental append -> +1000 rows (table total = 2002)


## Validation & Audit Trail

In [0]:
summary_df = spark.createDataFrame(bronze_summary, ["entity", "batch", "row_count"])
display(summary_df)

for entity in ENTITIES:
    table = bronze_table_name(entity)
    df = spark.table(table)
    assert "ingested_at" in df.columns, f"{table} missing ingested_at metadata column!"
    print(f"✅ {table}: ingested_at present, total rows = {df.count()}")

print("Bronze layer complete. Tables:", [bronze_table_name(e) for e in ENTITIES])

entity,batch,row_count
customer,historical,1052
customer,incremental_appended,1053
product,historical,1043
product,incremental_appended,1041
sales,historical,1002
sales,incremental_appended,1000


✅ apex_retail1.bronze_tables.bronze_customer: ingested_at present, total rows = 2105
✅ apex_retail1.bronze_tables.bronze_product: ingested_at present, total rows = 2084
✅ apex_retail1.bronze_tables.bronze_sales: ingested_at present, total rows = 2002
Bronze layer complete. Tables: ['apex_retail1.bronze_tables.bronze_customer', 'apex_retail1.bronze_tables.bronze_product', 'apex_retail1.bronze_tables.bronze_sales']


### Idempotency Note
Re-running the **historical** cell is intentionally `overwrite` (safe to re-run without
duplication). Re-running the **incremental** cell with `mode("append")` on the *same*
incremental batch would duplicate rows in Bronze by design — Bronze is a raw append log.
True idempotency (no duplicate business records) is enforced at the **Silver** layer via
`MERGE INTO` semantics in `03_silver_layer.py`, which is where re-run safety actually matters
for reporting correctness.